# Proyecto final – Segmentación semántica de mascotas con U-Net

**Suficiencia – Introducción al Aprendizaje Profundo**
**Estudiante:** Diego Alexander Valencia Calderón
**Dataset:** Oxford-IIIT Pet (máscaras de segmentación *trimap*)

**Objetivo:** construir un modelo que, dada la fotografía de un perro o un gato, asigne a **cada píxel** una de tres clases: mascota, fondo o contorno. Se usa una **U-Net** con un encoder **MobileNetV2 preentrenado en ImageNet**, y el desempeño se mide con métricas propias de segmentación (IoU y Dice).

> **Entorno recomendado:** Google Colab con GPU T4.

| Sección | Requisito del enunciado |
|---|---|
| 1 | a) Definición del problema |
| 2 | b) Descripción y análisis del conjunto de datos |
| 3 | c) Preparación de los datos |
| 4 | d) Selección de la arquitectura |
| 5 | e) Entrenamiento del modelo |
| 6 | f) Evaluación del modelo |
| 7 | g) Análisis de resultados |
| 8 | Resumen de métricas y conclusiones |

**Salidas generadas**

| Archivo | Contenido |
|---|---|
| `figuras/01_ejemplos_dataset.png` | Imágenes con su máscara real |
| `figuras/02_distribucion_pixeles.png` | Distribución de píxeles por clase y tamaño de la mascota |
| `figuras/03_aumento_datos.png` | Aumento de datos sincronizado imagen–máscara |
| `figuras/04_curvas_entrenamiento.png` | Curvas de pérdida, IoU y accuracy (fases 1 y 2) |
| `figuras/05_metricas_por_clase.png` | IoU y Dice por clase |
| `figuras/06_matriz_confusion_pixeles.png` | Matriz de confusión a nivel de píxel |
| `figuras/07_predicciones_correctas.png` | Ejemplos con mejor IoU |
| `figuras/08_predicciones_errores.png` | Ejemplos con peor IoU |
| `figuras/09_distribucion_iou.png` | Distribución del IoU por imagen |
| `modelos/unet_fase1_encoder_congelado.keras` | Mejor modelo de la fase 1 |
| `modelos/unet_fase2_fine_tuning.keras` | Mejor modelo de la fase 2 |
| `resultados/history_fase1.csv`, `history_fase2.csv` | Historial de entrenamiento |
| `resultados/metricas_por_clase.csv` | IoU y Dice por clase en prueba |
| `resultados/metricas_proyecto.json` | Resumen de métricas del proyecto |

## 0. Configuración del entorno

Detecta si el notebook corre en Colab o en local, define las rutas de salida, fija la semilla para reproducibilidad y verifica la GPU.

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
NOTEBOOK = "proyecto_final/proyecto_fase2_segmentacion.ipynb"  # ruta de este notebook dentro del proyecto

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Busca la carpeta del proyecto en "Mi unidad" (hasta 4 niveles), sin importar dónde se haya subido
    MI_UNIDAD = Path("/content/drive/MyDrive")
    REPO_DIR = next(
        (
            ruta.parents[1]
            for nivel in ("*", "*/*", "*/*/*", "*/*/*/*")
            for ruta in MI_UNIDAD.glob(f"{nivel}/{NOTEBOOK}")
        ),
        MI_UNIDAD / "Suficiencia_Aprendizaje_Profundo",
    )
else:
    # En local se sube desde el directorio actual hasta encontrar la carpeta que contiene este notebook
    REPO_DIR = next(
        (carpeta for carpeta in (Path.cwd(), *Path.cwd().parents) if (carpeta / NOTEBOOK).exists()),
        Path.cwd().parent,
    )

assert REPO_DIR.exists(), f"No se encontró la carpeta del proyecto en {REPO_DIR}"

PROYECTO_DIR = REPO_DIR / "proyecto_final"
FIG_DIR = PROYECTO_DIR / "figuras"
RES_DIR = PROYECTO_DIR / "resultados"
MOD_DIR = PROYECTO_DIR / "modelos"
for carpeta in (FIG_DIR, RES_DIR, MOD_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)  # fija las semillas de Python, NumPy y TensorFlow

print("Entorno:", "Google Colab" if EN_COLAB else "Local")
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "no disponible")
print("Repositorio:", REPO_DIR)

## 1. Definición del problema (a)

**Contexto.** Separar automáticamente a una mascota del fondo de una fotografía es útil en aplicaciones de edición de fotos (quitar o cambiar el fondo), en plataformas de adopción y tiendas que necesitan recortes limpios, y como paso previo para otros sistemas: por ejemplo, un clasificador de razas puede trabajar mejor si solo ve al animal. En el Taller 2 se clasificó la **raza** de la imagen completa; este proyecto da el paso siguiente: entender la imagen **a nivel de píxel**.

**Tipo de tarea.** Segmentación **semántica**: cada píxel recibe una clase, sin distinguir entre instancias (el dataset tiene una mascota por imagen).

| Elemento | Descripción |
|---|---|
| **Entrada** | Fotografía RGB de un perro o gato, redimensionada a 160 × 160 × 3 |
| **Salida** | Máscara de 160 × 160 en la que cada píxel tiene una de 3 clases |
| **Clase 0 – Mascota** | Píxeles que pertenecen al animal |
| **Clase 1 – Fondo** | Píxeles que no pertenecen al animal |
| **Clase 2 – Contorno** | Franja de transición entre mascota y fondo (zona ambigua del borde) |

**Criterio de éxito.** Como el fondo ocupa buena parte de cada imagen, la exactitud por píxel es engañosa: un modelo que respondiera "fondo" en todas partes ya tendría una exactitud alta sin servir para nada. Por eso las métricas principales son el **IoU (Jaccard)** y el **coeficiente Dice** por clase, y su promedio (mIoU).

## 2. Descripción y análisis del conjunto de datos (b)

### 2.1 Parámetros generales

Se fijan la resolución, el tamaño del lote, la partición de validación y la definición de las clases. Esta celda también instala `importlib_resources` si falta: `tensorflow_datasets` lo necesita para descargar el dataset y algunas versiones de Colab no lo incluyen.

In [ ]:
import importlib
import importlib.util
import subprocess

# tensorflow_datasets necesita el paquete importlib_resources al descargar el dataset,
# pero algunas imágenes de Colab (Python 3.13) no lo traen instalado: se instala solo si falta
if importlib.util.find_spec("importlib_resources") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "importlib_resources"], check=True)
    importlib.invalidate_caches()

import time
import matplotlib.colors
from tensorflow.keras.applications import mobilenet_v2

TAM_IMAGEN = 160  # resolución soportada por los pesos de ImageNet de MobileNetV2 (96-224)
FORMA_ENTRADA = (TAM_IMAGEN, TAM_IMAGEN, 3)
TAMANO_BATCH = 32
PROPORCION_VAL = 0.15
AUTOTUNE = tf.data.AUTOTUNE

CLASES = ["Mascota", "Fondo", "Contorno"]
NUM_CLASES = len(CLASES)
COLORES = ["#f58518", "#1f3b73", "#f2cf5b"]
CMAP_MASCARA = matplotlib.colors.ListedColormap(COLORES)

print(f"Imágenes {TAM_IMAGEN}x{TAM_IMAGEN} | batch {TAMANO_BATCH} | clases: {CLASES}")

### 2.2 Carga y partición

Se usa el mismo dataset y la misma partición que en el Taller 2:

| Conjunto | Origen | Uso |
|---|---|---|
| Entrenamiento | 85 % del split `train` | Ajustar los pesos |
| Validación | 15 % del split `train` | *Early stopping*, reducción de la tasa de aprendizaje y selección del mejor modelo |
| Prueba | Split `test` completo | Evaluación final |

La partición está estratificada por raza y es reproducible con la semilla 42. En el dataset, la máscara viene codificada con 1 = mascota, 2 = fondo y 3 = contorno.

In [ ]:
from sklearn.model_selection import train_test_split

(ds_train_completo, ds_test_crudo), info = tfds.load(
    "oxford_iiit_pet", split=["train", "test"], with_info=True, shuffle_files=False
)
print("Versión del dataset:", info.version)
print("Splits originales:", {nombre: split.num_examples for nombre, split in info.splits.items()})

# Lectura rápida de las razas (sin decodificar imágenes) para estratificar la partición
DECODIFICADORES = {"image": tfds.decode.SkipDecoding(), "segmentation_mask": tfds.decode.SkipDecoding()}
ds_etiquetas = tfds.load("oxford_iiit_pet", split="train", decoders=DECODIFICADORES, shuffle_files=False)
razas_train = np.array([int(ej["label"]) for ej in tfds.as_numpy(ds_etiquetas)])

indices = np.arange(len(razas_train))
idx_train, idx_val = train_test_split(indices, test_size=PROPORCION_VAL, stratify=razas_train, random_state=SEMILLA)
mascara_train = tf.constant(np.isin(indices, idx_train))


def filtrar(ds, en_train):
    return (
        ds.enumerate()
        .filter(lambda i, ejemplo: tf.equal(tf.gather(mascara_train, i), en_train))
        .map(lambda i, ejemplo: ejemplo)
    )


ds_train_crudo = filtrar(ds_train_completo, True)
ds_val_crudo = filtrar(ds_train_completo, False)

N_TRAIN, N_VAL, N_TEST = len(idx_train), len(idx_val), info.splits["test"].num_examples
resumen = pd.DataFrame(
    {"Imágenes": [N_TRAIN, N_VAL, N_TEST]}, index=pd.Index(["Entrenamiento", "Validación", "Prueba"], name="Conjunto")
)
resumen["Porcentaje (%)"] = (resumen["Imágenes"] / resumen["Imágenes"].sum() * 100).round(2)
display(resumen)

### 2.3 Ejemplos representativos

Cada fila muestra la fotografía original, su máscara (*trimap*) y la superposición de ambas. El contorno es una franja delgada alrededor del animal: corresponde a los píxeles del borde donde no es claro si pertenecen a la mascota o al fondo (pelo, sombras, desenfoque).

In [ ]:
ejemplos = list(tfds.as_numpy(ds_train_crudo.take(5)))
fig, axes = plt.subplots(len(ejemplos), 3, figsize=(11, 3.4 * len(ejemplos)))
for fila, ejemplo in zip(axes, ejemplos):
    mascara = ejemplo["segmentation_mask"][..., 0] - 1  # 1, 2, 3 -> 0, 1, 2
    fila[0].imshow(ejemplo["image"])
    fila[0].set_title(f"Imagen ({ejemplo['image'].shape[0]}x{ejemplo['image'].shape[1]})")
    fila[1].imshow(mascara, cmap=CMAP_MASCARA, vmin=0, vmax=NUM_CLASES - 1, interpolation="nearest")
    fila[1].set_title("Máscara real")
    fila[2].imshow(ejemplo["image"])
    fila[2].imshow(mascara, cmap=CMAP_MASCARA, vmin=0, vmax=NUM_CLASES - 1, alpha=0.45, interpolation="nearest")
    fila[2].set_title("Superposición")
    for ax in fila:
        ax.axis("off")
leyenda = [plt.Rectangle((0, 0), 1, 1, color=c) for c in COLORES]
fig.legend(leyenda, CLASES, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.01), frameon=False, fontsize=12)
fig.suptitle("Ejemplos del conjunto de entrenamiento", fontsize=14, y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_ejemplos_dataset.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.4 Distribución de los píxeles por clase

Se cuentan los píxeles de cada clase en todas las máscaras de entrenamiento (en su resolución original) y la fracción de la imagen que ocupa la mascota. Esto muestra el **desbalance de clases** y la variedad de tamaños del objeto a segmentar.

In [ ]:
conteo_pixeles = np.zeros(NUM_CLASES, dtype=np.int64)
fraccion_mascota, tamanos = [], []
for mascara in tfds.as_numpy(ds_train_crudo.map(lambda ej: ej["segmentation_mask"])):
    valores = mascara[..., 0].astype(np.int64) - 1
    cuenta = np.bincount(valores.ravel(), minlength=NUM_CLASES)
    conteo_pixeles += cuenta
    fraccion_mascota.append(cuenta[0] / valores.size)
    tamanos.append(valores.shape)
fraccion_mascota = np.array(fraccion_mascota)
tamanos = np.array(tamanos)

distribucion = pd.DataFrame(
    {"Píxeles": conteo_pixeles, "Porcentaje (%)": (conteo_pixeles / conteo_pixeles.sum() * 100).round(2)},
    index=pd.Index(CLASES, name="Clase"),
)
display(distribucion)
print(f"Fracción de la imagen ocupada por la mascota: media {fraccion_mascota.mean():.1%}, "
      f"mínimo {fraccion_mascota.min():.1%}, máximo {fraccion_mascota.max():.1%}")
print(f"Tamaño original de las imágenes: alto medio {tamanos[:, 0].mean():.0f} px, ancho medio {tamanos[:, 1].mean():.0f} px")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
barras = axes[0].bar(CLASES, distribucion["Porcentaje (%)"], color=COLORES, edgecolor="black")
for barra, valor in zip(barras, distribucion["Porcentaje (%)"]):
    axes[0].text(barra.get_x() + barra.get_width() / 2, valor + 1, f"{valor:.1f} %", ha="center", fontsize=11)
axes[0].set_ylabel("Porcentaje de píxeles (%)")
axes[0].set_ylim(0, distribucion["Porcentaje (%)"].max() * 1.18)
axes[0].set_title("Píxeles por clase (entrenamiento)")
axes[1].hist(fraccion_mascota * 100, bins=30, color=COLORES[0], edgecolor="black")
axes[1].axvline(fraccion_mascota.mean() * 100, color="black", linestyle="--", label=f"Media = {fraccion_mascota.mean():.1%}")
axes[1].set_xlabel("Porcentaje de la imagen ocupado por la mascota (%)")
axes[1].set_ylabel("Número de imágenes")
axes[1].set_title("Tamaño relativo de la mascota por imagen")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "02_distribucion_pixeles.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Preparación de los datos (c)

| Paso | Imagen | Máscara | Justificación |
|---|---|---|---|
| Redimensionado a 160 × 160 | Interpolación bilineal | Vecino más cercano | La máscara debe conservar valores enteros 0, 1, 2; interpolar crearía clases inexistentes |
| Recodificación | — | 1, 2, 3 → 0, 1, 2 | Las pérdidas de Keras esperan clases desde 0 |
| Caché en memoria | `uint8` | `uint8` | Se decodifica y redimensiona una sola vez |
| Aumento de datos (solo entrenamiento) | Volteo + brillo + contraste | Mismo volteo | Más variedad sin cambiar la clase de cada píxel |
| Normalización | `mobilenet_v2.preprocess_input` (rango [−1, 1]) | — | Es la escala con la que se entrenó MobileNetV2 en ImageNet |
| Lotes y *prefetch* | 32 imágenes | 32 máscaras | Uso eficiente de la GPU |

**Aumento sincronizado.** El volteo horizontal debe aplicarse **igual** a la imagen y a su máscara; si no, la máscara dejaría de corresponder a la foto. Para garantizarlo, imagen y máscara se concatenan en un solo tensor de 4 canales, se voltean juntas y luego se separan. Los cambios de brillo y contraste se aplican solo a la imagen, porque no alteran la clase de ningún píxel.

A diferencia de EfficientNet (Taller 2), MobileNetV2 **no** incluye la normalización dentro del modelo, así que aquí sí se aplica `preprocess_input`, que lleva los píxeles de [0, 255] a [−1, 1].

In [ ]:
def redimensionar(ejemplo):
    imagen = tf.image.resize(ejemplo["image"], (TAM_IMAGEN, TAM_IMAGEN), method="bilinear")
    imagen = tf.cast(tf.clip_by_value(tf.round(imagen), 0, 255), tf.uint8)
    mascara = tf.image.resize(ejemplo["segmentation_mask"], (TAM_IMAGEN, TAM_IMAGEN), method="nearest")
    return imagen, tf.cast(mascara, tf.uint8) - 1  # 1, 2, 3 -> 0, 1, 2


def aumentar(imagen, mascara):
    # El volteo se aplica sobre imagen y máscara concatenadas para que sea idéntico en ambas
    combinado = tf.concat([tf.cast(imagen, tf.float32), tf.cast(mascara, tf.float32)], axis=-1)
    combinado = tf.image.random_flip_left_right(combinado)
    imagen, mascara = combinado[..., :3], combinado[..., 3:]
    imagen = tf.image.random_brightness(imagen, max_delta=25.0)
    imagen = tf.image.random_contrast(imagen, 0.8, 1.2)
    return tf.clip_by_value(imagen, 0.0, 255.0), tf.cast(mascara, tf.uint8)


def preprocesar(imagenes, mascaras):
    return mobilenet_v2.preprocess_input(tf.cast(imagenes, tf.float32)), tf.cast(mascaras, tf.int32)


def crear_pipeline(ds_crudo, entrenamiento=False):
    ds = ds_crudo.map(redimensionar, num_parallel_calls=AUTOTUNE).cache()
    if entrenamiento:
        ds = ds.shuffle(N_TRAIN, seed=SEMILLA, reshuffle_each_iteration=True).map(aumentar, num_parallel_calls=AUTOTUNE)
    return ds.batch(TAMANO_BATCH).map(preprocesar, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


ds_train = crear_pipeline(ds_train_crudo, entrenamiento=True)
ds_val = crear_pipeline(ds_val_crudo)
ds_test = crear_pipeline(ds_test_crudo)

imagenes, mascaras = next(iter(ds_val))
print("Lote de imágenes:", imagenes.shape, imagenes.dtype,
      f"| rango [{float(tf.reduce_min(imagenes)):.2f}, {float(tf.reduce_max(imagenes)):.2f}]")
print("Lote de máscaras:", mascaras.shape, mascaras.dtype, "| valores:", np.unique(mascaras.numpy()))

# Conjunto de prueba en memoria (sin normalizar) para las visualizaciones de la sección 7
X_test_img, Y_test = map(np.stack, zip(*tfds.as_numpy(ds_test_crudo.map(redimensionar, num_parallel_calls=AUTOTUNE))))
print("Prueba en memoria:", X_test_img.shape, Y_test.shape)

In [ ]:
# Efecto del aumento de datos: la misma imagen y su máscara transformadas varias veces
imagen_ej, mascara_ej = redimensionar(next(iter(ds_train_crudo.take(1))))
fig, axes = plt.subplots(2, 5, figsize=(16, 6.8))
for j in range(5):
    imagen_a, mascara_a = (tf.cast(imagen_ej, tf.float32), mascara_ej) if j == 0 else aumentar(imagen_ej, mascara_ej)
    axes[0, j].imshow(imagen_a.numpy().astype("uint8"))
    axes[0, j].set_title("Original" if j == 0 else f"Aumentada {j}", fontweight="bold" if j == 0 else None)
    axes[1, j].imshow(mascara_a.numpy()[..., 0], cmap=CMAP_MASCARA, vmin=0, vmax=NUM_CLASES - 1, interpolation="nearest")
    for ax in axes[:, j]:
        ax.axis("off")
fig.suptitle("Aumento de datos sincronizado: la máscara se voltea junto con la imagen", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "03_aumento_datos.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Selección de la arquitectura (d)

**U-Net** es la arquitectura de referencia para segmentación semántica. Tiene forma de "U":

- **Encoder (camino de bajada):** reduce progresivamente la resolución y extrae características cada vez más abstractas (*qué* hay en la imagen).
- **Decoder (camino de subida):** recupera la resolución original para producir una predicción por píxel (*dónde* está).
- **Conexiones de salto (*skip connections*):** llevan los mapas de alta resolución del encoder directamente al decoder. Sin ellas, los bordes de la máscara quedarían borrosos, porque el detalle fino se pierde al bajar a 5 × 5 píxeles.

**Encoder: MobileNetV2 preentrenada en ImageNet.** Con unas 3.100 imágenes de entrenamiento, aprender un encoder desde cero sería poco eficiente; el Taller 2 mostró que reutilizar pesos de ImageNet es decisivo con pocos datos. MobileNetV2 es ligera (convoluciones *depthwise separable*), lo que permite entrenar rápido en la GPU de Colab. Se toman como salidas cinco capas internas:

| Capa de MobileNetV2 | Resolución | Uso |
|---|---|---|
| `block_1_expand_relu` | 80 × 80 | Conexión de salto 1 |
| `block_3_expand_relu` | 40 × 40 | Conexión de salto 2 |
| `block_6_expand_relu` | 20 × 20 | Conexión de salto 3 |
| `block_13_expand_relu` | 10 × 10 | Conexión de salto 4 |
| `block_16_project` | 5 × 5 | Cuello de botella |

**Decoder.** Cuatro bloques, cada uno con: `Conv2DTranspose` (duplica la resolución) → BatchNormalization → ReLU → concatenación con la conexión de salto → `Conv2D 3×3` → BatchNormalization → ReLU, con 256, 128, 64 y 32 filtros. Una última `Conv2DTranspose` lleva de 80 × 80 a 160 × 160 con 3 canales y activación *softmax*: la probabilidad de cada clase en cada píxel.

Como en el Taller 2, el encoder se llama con `training=False` para que sus capas BatchNormalization conserven las estadísticas de ImageNet también durante el *fine-tuning*.

In [ ]:
CAPAS_SALTO = ["block_1_expand_relu", "block_3_expand_relu", "block_6_expand_relu", "block_13_expand_relu", "block_16_project"]
FILTROS_DECODER = [256, 128, 64, 32]


def bloque_decoder(x, salto, filtros, nombre):
    x = keras.layers.Conv2DTranspose(filtros, 3, strides=2, padding="same", use_bias=False, name=f"{nombre}_subida")(x)
    x = keras.layers.BatchNormalization(name=f"{nombre}_bn1")(x)
    x = keras.layers.ReLU(name=f"{nombre}_relu1")(x)
    x = keras.layers.Concatenate(name=f"{nombre}_salto")([x, salto])
    x = keras.layers.Conv2D(filtros, 3, padding="same", use_bias=False, name=f"{nombre}_conv")(x)
    x = keras.layers.BatchNormalization(name=f"{nombre}_bn2")(x)
    return keras.layers.ReLU(name=f"{nombre}_relu2")(x)


def construir_unet():
    base = keras.applications.MobileNetV2(input_shape=FORMA_ENTRADA, include_top=False, weights="imagenet")
    encoder = keras.Model(base.input, [base.get_layer(nombre).output for nombre in CAPAS_SALTO], name="encoder_mobilenetv2")
    encoder.trainable = False

    entradas = keras.Input(shape=FORMA_ENTRADA, name="imagen")
    *saltos, x = encoder(entradas, training=False)  # BatchNormalization del encoder en modo inferencia
    for i, (filtros, salto) in enumerate(zip(FILTROS_DECODER, reversed(saltos)), start=1):
        x = bloque_decoder(x, salto, filtros, f"decoder{i}")
    salidas = keras.layers.Conv2DTranspose(NUM_CLASES, 3, strides=2, padding="same", activation="softmax", name="mascara")(x)
    return keras.Model(entradas, salidas, name="unet_mobilenetv2")


def contar_parametros(modelo):
    entrenables = int(sum(np.prod(w.shape) for w in modelo.trainable_weights))
    return int(modelo.count_params()), entrenables


modelo = construir_unet()
encoder = modelo.get_layer("encoder_mobilenetv2")
print("Salidas del encoder:", [tuple(t.shape[1:]) for t in encoder.outputs])
total, entrenables = contar_parametros(modelo)
print(f"Parámetros totales: {total:,} | encoder: {encoder.count_params():,} | entrenables en la fase 1: {entrenables:,}")
modelo.summary(line_length=110)

## 5. Entrenamiento del modelo (e)

### 5.1 Decisiones de entrenamiento

| Elemento | Elección | Justificación |
|---|---|---|
| **Función de pérdida** | Entropía cruzada + pérdida Dice | La entropía cruzada da gradientes estables píxel a píxel, pero está dominada por la clase mayoritaria (fondo). La pérdida Dice mide el solapamiento de cada clase sin importar su tamaño, así que obliga al modelo a atender también al contorno, que es la clase minoritaria |
| **Optimizador** | Adam | Tasa de aprendizaje adaptativa por parámetro; converge rápido y es robusto |
| **Tasa de aprendizaje** | 0,001 en la fase 1; 0,0001 en la fase 2 | En la fase 1 solo se entrena el decoder, que parte de pesos aleatorios. En la fase 2 se ajusta también el encoder preentrenado, con pasos 10 veces más pequeños para no destruir lo aprendido en ImageNet |
| **Épocas** | Máximo 25 (fase 1) y 20 (fase 2) | Son techos: el número real lo decide *early stopping* |
| **Tamaño del batch** | 32 | Equilibrio entre estabilidad del gradiente y memoria de la GPU T4 a 160 × 160 |
| **Métrica de control** | mIoU de validación | Es la métrica objetivo del problema |

**Estrategias para controlar y mejorar el entrenamiento:**

1. **Transfer Learning en dos fases:** primero se entrena solo el decoder con el encoder congelado; luego se descongela el encoder (sin sus BatchNormalization) para ajustarlo a la tarea.
2. **EarlyStopping** sobre el mIoU de validación, con restauración de los mejores pesos.
3. **ModelCheckpoint:** guarda el mejor modelo de cada fase.
4. **ReduceLROnPlateau:** reduce la tasa de aprendizaje a la mitad si la pérdida de validación se estanca 2 épocas.
5. **Aumento de datos sincronizado** (sección 3) para reducir el sobreajuste.
6. **BatchNormalization** en el decoder para estabilizar el entrenamiento.

In [ ]:
def perdida_ce_dice(y_true, y_pred):
    # Entropía cruzada por píxel + (1 - Dice promedio de las clases), calculada por imagen
    y_true = tf.cast(tf.reshape(y_true, tf.shape(y_pred)[:-1]), tf.int32)
    entropia = keras.losses.sparse_categorical_crossentropy(y_true, y_pred)  # (B, H, W)
    y_one_hot = tf.one_hot(y_true, NUM_CLASES)
    interseccion = tf.reduce_sum(y_one_hot * y_pred, axis=[1, 2])
    suma = tf.reduce_sum(y_one_hot + y_pred, axis=[1, 2])
    dice = (2.0 * interseccion + 1.0) / (suma + 1.0)  # (B, C), con suavizado para evitar divisiones por 0
    return tf.reduce_mean(entropia, axis=[1, 2]) + (1.0 - tf.reduce_mean(dice, axis=-1))


def compilar(modelo, tasa_aprendizaje):
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=tasa_aprendizaje),
        loss=perdida_ce_dice,
        metrics=[
            keras.metrics.MeanIoU(num_classes=NUM_CLASES, sparse_y_true=True, sparse_y_pred=False, name="mean_iou"),
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        ],
    )


def callbacks_fase(archivo, paciencia):
    return [
        keras.callbacks.EarlyStopping(monitor="val_mean_iou", mode="max", patience=paciencia, restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(str(MOD_DIR / archivo), monitor="val_mean_iou", mode="max", save_best_only=True, verbose=0),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ]


def matriz_pixeles(modelo, ds):
    # Matriz de confusión acumulada sobre todos los píxeles del conjunto
    matriz = np.zeros((NUM_CLASES, NUM_CLASES), dtype=np.int64)
    for imagenes, mascaras in ds:
        pred = np.argmax(modelo.predict_on_batch(imagenes), axis=-1).ravel()
        real = mascaras.numpy().ravel()
        matriz += np.bincount(real * NUM_CLASES + pred, minlength=NUM_CLASES**2).reshape(NUM_CLASES, NUM_CLASES)
    return matriz


def metricas_desde_matriz(matriz):
    tp = np.diag(matriz).astype(float)
    fp = matriz.sum(axis=0) - tp
    fn = matriz.sum(axis=1) - tp
    iou = tp / (tp + fp + fn)
    dice = 2 * tp / (2 * tp + fp + fn)
    return {
        "iou_por_clase": dict(zip(CLASES, iou.round(4).tolist())),
        "dice_por_clase": dict(zip(CLASES, dice.round(4).tolist())),
        "mean_iou": float(iou.mean()),
        "mean_dice": float(dice.mean()),
        "pixel_accuracy": float(tp.sum() / matriz.sum()),
    }


fases = {}


def registrar_fase(clave, descripcion, history, archivo, duracion):
    historial = pd.DataFrame(history.history)
    historial.index = pd.RangeIndex(1, len(historial) + 1, name="época")
    historial.to_csv(RES_DIR / f"history_{clave}.csv")
    modelo.load_weights(MOD_DIR / archivo)  # mejor época según el mIoU de validación
    matriz_val, matriz_test = matriz_pixeles(modelo, ds_val), matriz_pixeles(modelo, ds_test)
    total, entrenables = contar_parametros(modelo)
    fases[clave] = {
        "descripcion": descripcion,
        "archivo": f"modelos/{archivo}",
        "parametros_entrenables": entrenables,
        "epocas_entrenadas": len(historial),
        "mejor_epoca": int(historial["val_mean_iou"].idxmax()),
        "tiempo_entrenamiento_s": round(duracion, 1),
        "validacion": metricas_desde_matriz(matriz_val),
        "prueba": metricas_desde_matriz(matriz_test),
        "matriz_prueba": matriz_test,
        "historial": historial,
    }
    v, p = fases[clave]["validacion"], fases[clave]["prueba"]
    print(f"\n{descripcion}: mIoU val = {v['mean_iou']:.4f} | mIoU prueba = {p['mean_iou']:.4f} | "
          f"Dice prueba = {p['mean_dice']:.4f} | épocas = {len(historial)} | tiempo = {duracion / 60:.1f} min")

### 5.2 Fase 1 – Encoder congelado

Solo se entrenan el decoder y la capa de salida. El encoder actúa como extractor de características fijo, igual que en el experimento B del Taller 2.

In [ ]:
EPOCAS_F1, PACIENCIA_F1, LR_F1 = 25, 5, 1e-3
ARCHIVO_F1 = "unet_fase1_encoder_congelado.keras"

compilar(modelo, LR_F1)
inicio = time.time()
history_f1 = modelo.fit(ds_train, validation_data=ds_val, epochs=EPOCAS_F1,
                        callbacks=callbacks_fase(ARCHIVO_F1, PACIENCIA_F1), verbose=2)
registrar_fase("fase1", "Fase 1 – encoder congelado", history_f1, ARCHIVO_F1, time.time() - inicio)

### 5.3 Fase 2 – Fine-tuning del encoder

Se parte del mejor modelo de la fase 1 y se descongela todo el encoder, excepto sus capas BatchNormalization. La tasa de aprendizaje baja a 0,0001 para ajustar los filtros de ImageNet a la tarea de segmentación sin borrar lo que ya saben.

In [ ]:
EPOCAS_F2, PACIENCIA_F2, LR_F2 = 20, 5, 1e-4
ARCHIVO_F2 = "unet_fase2_fine_tuning.keras"

encoder = modelo.get_layer("encoder_mobilenetv2")
encoder.trainable = True
for capa in encoder.layers:
    if isinstance(capa, keras.layers.BatchNormalization):
        capa.trainable = False
total, entrenables = contar_parametros(modelo)
print(f"Parámetros entrenables en la fase 2: {entrenables:,} de {total:,}")

compilar(modelo, LR_F2)
inicio = time.time()
history_f2 = modelo.fit(ds_train, validation_data=ds_val, epochs=EPOCAS_F2,
                        callbacks=callbacks_fase(ARCHIVO_F2, PACIENCIA_F2), verbose=2)
registrar_fase("fase2", "Fase 2 – fine-tuning", history_f2, ARCHIVO_F2, time.time() - inicio)

## 6. Evaluación del modelo (f)

### 6.1 Métricas

Todas las métricas se calculan a partir de la matriz de confusión **acumulada sobre todos los píxeles** del conjunto de prueba, con TP, FP y FN por clase:

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **IoU (Jaccard)** | TP / (TP + FP + FN) | Solapamiento entre la región predicha y la real; penaliza tanto lo que sobra como lo que falta |
| **Dice (F1)** | 2·TP / (2·TP + FP + FN) | Similar al IoU pero algo más indulgente; muy usado en segmentación |
| **mIoU / mDice** | Promedio de las 3 clases | Da el mismo peso a cada clase, sin importar su tamaño |
| **Exactitud por píxel** | Píxeles correctos / total | Se reporta solo como referencia: está inflada por el fondo |

Para mostrar por qué la exactitud por píxel engaña, se compara además con un **modelo trivial** que predice "fondo" en todos los píxeles.

El modelo final se elige por **mIoU de validación**, no de prueba, igual que en el Taller 2.

In [ ]:
MEJOR_FASE = max(fases, key=lambda f: fases[f]["validacion"]["mean_iou"])
modelo.load_weights(MOD_DIR / Path(fases[MEJOR_FASE]["archivo"]).name)
print(f"Modelo seleccionado (mayor mIoU de validación): {fases[MEJOR_FASE]['descripcion']}")

# Modelo trivial: predice "Fondo" en todos los píxeles
matriz_trivial = np.zeros((NUM_CLASES, NUM_CLASES), dtype=np.int64)
matriz_trivial[:, CLASES.index("Fondo")] = fases[MEJOR_FASE]["matriz_prueba"].sum(axis=1)
trivial = metricas_desde_matriz(matriz_trivial)

filas = []
for nombre, m in [
    ("Trivial (todo fondo)", trivial),
    (fases["fase1"]["descripcion"], fases["fase1"]["prueba"]),
    (fases["fase2"]["descripcion"], fases["fase2"]["prueba"]),
]:
    filas.append({"Modelo": nombre, "mIoU": m["mean_iou"], "mDice": m["mean_dice"], "Exactitud por píxel": m["pixel_accuracy"],
                  **{f"IoU {c}": m["iou_por_clase"][c] for c in CLASES}})
comparacion = pd.DataFrame(filas).set_index("Modelo")
display(comparacion.apply(lambda columna: columna.map("{:.2%}".format)))

metricas_clase = pd.DataFrame(
    {"IoU": fases[MEJOR_FASE]["prueba"]["iou_por_clase"], "Dice": fases[MEJOR_FASE]["prueba"]["dice_por_clase"]}
)
metricas_clase.index.name = "Clase"
metricas_clase.to_csv(RES_DIR / "metricas_por_clase.csv")

In [ ]:
# IoU y Dice por clase en prueba, fase 1 frente a fase 2
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
ancho = 0.38
x = np.arange(NUM_CLASES)
for ax, metrica, titulo in [(axes[0], "iou_por_clase", "IoU por clase"), (axes[1], "dice_por_clase", "Dice por clase")]:
    for k, (clave, color) in enumerate([("fase1", "#9ecae1"), ("fase2", "#3182bd")]):
        valores = [fases[clave]["prueba"][metrica][c] for c in CLASES]
        barras = ax.bar(x + (k - 0.5) * ancho, valores, ancho, color=color, edgecolor="black", label=fases[clave]["descripcion"])
        for barra, valor in zip(barras, valores):
            ax.text(barra.get_x() + barra.get_width() / 2, valor + 0.01, f"{valor:.2f}", ha="center", fontsize=9)
    ax.set_xticks(x, CLASES)
    ax.set_ylim(0, 1.08)
    ax.set_title(f"{titulo} (prueba)")
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Valor")
manijas, etiquetas = axes[0].get_legend_handles_labels()
fig.legend(manijas, etiquetas, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.07), frameon=False, fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "05_metricas_por_clase.png", dpi=150, bbox_inches="tight")
plt.show()

# Matriz de confusión a nivel de píxel del modelo seleccionado, normalizada por fila
import seaborn as sns

matriz = fases[MEJOR_FASE]["matriz_prueba"]
normalizada = matriz / matriz.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(7, 5.8))
sns.heatmap(normalizada, annot=True, fmt=".1%", cmap="Blues", xticklabels=CLASES, yticklabels=CLASES,
            cbar_kws={"label": "Proporción de píxeles de la clase real"}, ax=ax)
ax.set_xlabel("Clase predicha")
ax.set_ylabel("Clase real")
ax.set_title(f"Matriz de confusión por píxel – {fases[MEJOR_FASE]['descripcion']}")
fig.tight_layout()
fig.savefig(FIG_DIR / "06_matriz_confusion_pixeles.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Análisis de resultados (g)

### 7.1 Comportamiento durante el entrenamiento

Las curvas muestran las dos fases seguidas; la línea vertical marca el inicio del *fine-tuning*.

In [ ]:
import matplotlib.ticker

historial_total = pd.concat([fases["fase1"]["historial"], fases["fase2"]["historial"]], ignore_index=True)
historial_total.index = pd.RangeIndex(1, len(historial_total) + 1, name="época")
inicio_f2 = len(fases["fase1"]["historial"]) + 0.5

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, (metrica, titulo) in zip(axes, [("loss", "Pérdida (CE + Dice)"), ("mean_iou", "mIoU"), ("accuracy", "Exactitud por píxel")]):
    ax.plot(historial_total.index, historial_total[metrica], marker="o", markersize=3, label="Entrenamiento")
    ax.plot(historial_total.index, historial_total[f"val_{metrica}"], marker="o", markersize=3, label="Validación")
    ax.axvline(inicio_f2, color="gray", linestyle="--", label="Inicio fase 2")
    ax.set_title(titulo)
    ax.set_xlabel("Época")
    ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle("Curvas de entrenamiento de la U-Net (fase 1: encoder congelado | fase 2: fine-tuning)", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "04_curvas_entrenamiento.png", dpi=150, bbox_inches="tight")
plt.show()

for clave in fases:
    h = fases[clave]["historial"]
    print(f"{fases[clave]['descripcion']}: mejor época {fases[clave]['mejor_epoca']} de {len(h)} | "
          f"mIoU entrenamiento {h['mean_iou'].iloc[-1]:.4f} vs validación {h['val_mean_iou'].iloc[-1]:.4f} (última época)")

### 7.2 Desempeño por imagen en el conjunto de prueba

Además de las métricas globales, se calcula el **IoU de la mascota en cada imagen** de prueba. Esto permite ver si el modelo funciona de forma pareja o si hay imágenes donde falla por completo, y elegir ejemplos representativos de aciertos y errores.

In [ ]:
predicciones = np.concatenate(
    [np.argmax(modelo.predict_on_batch(imagenes), axis=-1).astype(np.uint8) for imagenes, _ in ds_test]
)
reales = Y_test[..., 0]
mascota_real, mascota_pred = reales == 0, predicciones == 0
iou_mascota = (mascota_real & mascota_pred).sum(axis=(1, 2)) / np.maximum((mascota_real | mascota_pred).sum(axis=(1, 2)), 1)

print(f"IoU de la mascota por imagen: media {iou_mascota.mean():.4f} | mediana {np.median(iou_mascota):.4f}")
for umbral in (0.9, 0.8, 0.5):
    print(f"  Imágenes con IoU de la mascota >= {umbral:.0%}: {(iou_mascota >= umbral).mean():.1%}")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(iou_mascota, bins=40, color=COLORES[0], edgecolor="black")
ax.axvline(iou_mascota.mean(), color="black", linestyle="--", label=f"Media = {iou_mascota.mean():.3f}")
ax.axvline(np.median(iou_mascota), color="red", linestyle=":", label=f"Mediana = {np.median(iou_mascota):.3f}")
ax.set_xlabel("IoU de la mascota en la imagen")
ax.set_ylabel("Número de imágenes")
ax.set_title(f"Distribución del IoU de la mascota por imagen ({len(iou_mascota)} imágenes de prueba)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "09_distribucion_iou.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.3 Ejemplos de predicciones correctas y de errores

Cada fila muestra: la imagen, la máscara real, la máscara predicha y un mapa de errores (en rojo los píxeles mal clasificados). Se muestran las imágenes con **mayor** y **menor** IoU de la mascota.

In [ ]:
def grilla_predicciones(indices, titulo, archivo):
    fig, axes = plt.subplots(len(indices), 4, figsize=(14, 3.5 * len(indices)))
    for fila, i in zip(np.atleast_2d(axes), indices):
        error = reales[i] != predicciones[i]
        fila[0].imshow(X_test_img[i])
        fila[0].set_title(f"Imagen de prueba #{i}")
        fila[1].imshow(reales[i], cmap=CMAP_MASCARA, vmin=0, vmax=NUM_CLASES - 1, interpolation="nearest")
        fila[1].set_title("Máscara real")
        fila[2].imshow(predicciones[i], cmap=CMAP_MASCARA, vmin=0, vmax=NUM_CLASES - 1, interpolation="nearest")
        fila[2].set_title(f"Predicción (IoU mascota = {iou_mascota[i]:.2f})")
        fila[3].imshow(X_test_img[i])
        fila[3].imshow(np.ma.masked_where(~error, error), cmap=matplotlib.colors.ListedColormap(["red"]), alpha=0.6)
        fila[3].set_title(f"Errores ({error.mean():.1%} de los píxeles)")
        for ax in fila:
            ax.axis("off")
    leyenda = [plt.Rectangle((0, 0), 1, 1, color=c) for c in COLORES] + [plt.Rectangle((0, 0), 1, 1, color="red")]
    fig.legend(leyenda, CLASES + ["Píxel mal clasificado"], loc="upper center", ncol=4,
               bbox_to_anchor=(0.5, 1.0), frameon=False, fontsize=11)
    fig.suptitle(titulo, fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / archivo, dpi=130, bbox_inches="tight")
    plt.show()


orden = np.argsort(iou_mascota)
grilla_predicciones(orden[::-1][:4], "Predicciones con mayor IoU de la mascota", "07_predicciones_correctas.png")
grilla_predicciones(orden[:4], "Predicciones con menor IoU de la mascota", "08_predicciones_errores.png")

### 7.4 Análisis crítico

> Esta sección se completa con las cifras de la ejecución: los valores concretos están en las tablas y figuras de las secciones 6 y 7.

**Comportamiento durante el entrenamiento.** En la fase 1 el decoder aprende rápido porque el encoder ya entrega características útiles de ImageNet. En la fase 2, el *fine-tuning* adapta esas características, pensadas para clasificar, a la tarea de ubicar bordes y regiones. Una brecha creciente entre el mIoU de entrenamiento y el de validación indicaría sobreajuste; el *early stopping* y la reducción de la tasa de aprendizaje están para controlarlo.

**Desempeño por clase.** Se espera que la mascota y el fondo, regiones grandes y bien definidas, tengan IoU altos, y que el **contorno** sea la clase más difícil. Es una franja de pocos píxeles de ancho, su posición exacta es ambigua incluso en la anotación humana, y cualquier desplazamiento de un par de píxeles lo convierte en error. Por eso el mIoU queda por debajo del IoU de las clases grandes.

**Exactitud por píxel frente a IoU.** La comparación con el modelo trivial muestra por qué la exactitud por píxel no sirve como métrica principal: predecir "fondo" en todas partes obtiene una exactitud cercana a la proporción de fondo, con un IoU de la mascota de 0.

**Errores del modelo.** Las imágenes con peor IoU suelen corresponder a mascotas con colores parecidos al fondo, mascotas muy pequeñas o parcialmente visibles, varias mascotas o personas en la foto, o iluminación extrema.

**Limitaciones.**

1. **Resolución de 160 × 160:** se pierde detalle en bordes finos (pelo, orejas, bigotes) y la evaluación se hace a esa resolución, no a la original.
2. **Anotaciones tipo *trimap*:** la clase contorno es una aproximación; el modelo no puede ser más preciso que la anotación.
3. **Una mascota por imagen:** el modelo no distingue entre varias mascotas (no es segmentación de instancias).
4. **Aumento de datos limitado** a volteo, brillo y contraste.

**Posibles mejoras.**

- Entrenar a mayor resolución (224 o 256 píxeles) y evaluar a la resolución original de cada imagen.
- Encoders más potentes (EfficientNet, ResNet50) o arquitecturas como DeepLabV3+.
- Aumento de datos geométrico sincronizado (rotaciones, recortes y escalados aplicados igual a imagen y máscara).
- Postprocesamiento (por ejemplo, conservar la región conectada más grande) para eliminar manchas aisladas.
- Ponderar más la clase contorno o usar una pérdida enfocada en bordes.

## 8. Resumen de métricas y conclusiones

Se exportan las métricas del proyecto a `resultados/metricas_proyecto.json`.

In [ ]:
import json

metricas = {
    "dataset": "Oxford-IIIT Pet (trimaps)",
    "tarea": "Segmentación semántica en 3 clases",
    "clases": CLASES,
    "conjuntos": {"entrenamiento": N_TRAIN, "validacion": N_VAL, "prueba": N_TEST},
    "arquitectura": "U-Net con encoder MobileNetV2 (ImageNet)",
    "configuracion": {
        "tam_imagen": TAM_IMAGEN, "batch_size": TAMANO_BATCH, "perdida": "entropía cruzada + Dice",
        "optimizador": "Adam", "lr_fase1": LR_F1, "lr_fase2": LR_F2,
        "epocas_maximas": {"fase1": EPOCAS_F1, "fase2": EPOCAS_F2}, "semilla": SEMILLA,
    },
    "fases": {clave: {k: v for k, v in datos.items() if k not in ("matriz_prueba", "historial")} for clave, datos in fases.items()},
    "modelo_seleccionado": MEJOR_FASE,
    "criterio_seleccion": "mayor mIoU de validación",
    "prueba_modelo_seleccionado": fases[MEJOR_FASE]["prueba"],
    "modelo_trivial_todo_fondo": trivial,
    "iou_mascota_por_imagen": {
        "media": float(iou_mascota.mean()), "mediana": float(np.median(iou_mascota)),
        "porcentaje_imagenes_iou_mayor_0_9": float((iou_mascota >= 0.9).mean()),
    },
}
(RES_DIR / "metricas_proyecto.json").write_text(json.dumps(metricas, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(metricas, ensure_ascii=False, indent=2))

### Conclusiones

1. Una **U-Net con encoder MobileNetV2 preentrenado** resuelve la segmentación de mascotas en tres clases con pocos datos (unas 3.100 imágenes de entrenamiento), gracias al *Transfer Learning* que ya se había comprobado en el Taller 2.
2. El entrenamiento en **dos fases** (decoder primero, *fine-tuning* después) permite aprovechar los pesos de ImageNet sin destruirlos; la comparación entre fases muestra cuánto aporta adaptar el encoder.
3. Las métricas de segmentación (IoU y Dice por clase) dan una imagen mucho más honesta que la exactitud por píxel, como demuestra el modelo trivial.
4. El **contorno** es la clase más difícil: es delgado, ambiguo y sensible a desplazamientos de pocos píxeles. Mejorar la resolución y el aumento de datos geométrico son las vías más directas para mejorarlo.